# Boxplots de IGD e HV: frentes completas e cardinalidade equalizada

Este notebook compara C-NBI, VRF-NBI, NSGA-III e MOEA/D nos nove cenários sintéticos. Cada caixa resume as dez repetições da campanha FULL e os dez resultados individuais são sobrepostos discretamente. Para cada métrica, a linha superior apresenta as frentes completas e a linha inferior apresenta as frentes com cardinalidade equalizada por KMeans-medoid.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator

def project_root(start=Path.cwd()):
    path = start.resolve()
    for candidate in (path, *path.parents):
        if (candidate / 'configs' / 'full.json').exists():
            return candidate
    raise FileNotFoundError('Raiz do projeto não encontrada.')

ROOT = project_root()
SOURCE_PATH = ROOT / 'results' / 'synthetic' / 'tables' / 'full_metrics.csv'
OUT_DIR = ROOT / 'results' / 'synthetic' / 'figures' / 'performance_boxplots'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_WIDTH_CM = 16.0
CM_TO_INCH = 1 / 2.54

METHODS = ['CNBI', 'VRF-NBI', 'NSGA-III', 'MOEA/D']
METHOD_COLORS = {
    'CNBI': '#D55E00',
    'VRF-NBI': '#009E73',
    'NSGA-III': '#CC79A7',
    'MOEA/D': '#E69F00',
}
METHOD_LABELS = {'CNBI': 'C-NBI', 'VRF-NBI': 'VRF-NBI', 'NSGA-III': 'NSGA-III', 'MOEA/D': 'MOEA/D'}
M_VALUES = [4, 6, 12]
CORRELATIONS = ['low', 'medium', 'high']
CORRELATION_LABELS = ['baixa', 'média', 'alta']
COMPARISONS = ['complete', 'equal_cardinality']
ROW_LABELS = {
    'complete': 'Fronteira completa',
    'equal_cardinality': 'Cardinalidade equalizada',
}

plt.rcParams.update({
    'font.family': 'DejaVu Serif',
    'font.size': 8.5,
    'axes.titlesize': 9.5,
    'axes.labelsize': 9.0,
    'xtick.labelsize': 7.5,
    'ytick.labelsize': 7.5,
    'legend.fontsize': 7.8,
    'savefig.facecolor': 'white',
    'axes.facecolor': 'white',
})
print('Fonte:', SOURCE_PATH.relative_to(ROOT))
print('Saída:', OUT_DIR.relative_to(ROOT))

In [ ]:
raw = pd.read_csv(SOURCE_PATH)
data = raw.loc[
    raw['method'].isin(METHODS) & raw['comparison'].isin(COMPARISONS)
].copy()
data['m'] = data['scenario'].str.extract(r'^m(\d+)_')[0].astype(int)
data['correlation'] = data['scenario'].str.extract(r'^m\d+_(.+)$')[0]

assert len(data) == 2 * 9 * 4 * 10
assert set(data['m']) == set(M_VALUES)
assert set(data['correlation']) == set(CORRELATIONS)
assert data[['IGD', 'HV']].notna().all().all()
group_sizes = data.groupby(['comparison', 'scenario', 'method']).size()
assert group_sizes.eq(10).all()
assert set(data.loc[data['comparison'].eq('equal_cardinality'), 'equalization_method']) == {'KMeans-medoid'}

source_export = OUT_DIR / 'boxplots_igd_hv_source.csv'
data.to_csv(source_export, index=False)

summary_parts = []
for metric in ['IGD', 'HV']:
    part = (
        data.groupby(['comparison', 'scenario', 'm', 'correlation', 'method'])[metric]
            .agg(n='size', minimum='min', q1=lambda x: x.quantile(0.25),
                 median='median', q3=lambda x: x.quantile(0.75), maximum='max')
            .reset_index()
    )
    part.insert(0, 'metric', metric)
    summary_parts.append(part)
summary = pd.concat(summary_parts, ignore_index=True)
summary_path = OUT_DIR / 'boxplots_igd_hv_summary.csv'
summary.to_csv(summary_path, index=False)
print('Linhas usadas:', len(data))
print('Grupos de dez repetições:', len(group_sizes))

In [ ]:
GROUP_CENTERS = np.arange(len(CORRELATIONS), dtype=float)
METHOD_OFFSETS = dict(zip(METHODS, [-0.27, -0.09, 0.09, 0.27]))
BOX_WIDTH = 0.15

def stable_jitter_key(*parts):
    text = '|'.join(map(str, parts))
    return sum((index + 1) * ord(character) for index, character in enumerate(text))

def draw_metric(metric, ylabel, stem):
    fig, axes = plt.subplots(
        2, 3, sharey=True,
        figsize=(FIGURE_WIDTH_CM * CM_TO_INCH, 14.2 * CM_TO_INCH),
    )
    fig.subplots_adjust(
        left=0.08, right=0.995, top=0.88, bottom=0.15,
        hspace=0.48, wspace=0.15,
    )

    global_max = float(data[metric].max())
    y_upper = global_max * 1.06

    panel_index = 0
    for row, comparison in enumerate(COMPARISONS):
        for column, m in enumerate(M_VALUES):
            ax = axes[row, column]
            for correlation_index, correlation in enumerate(CORRELATIONS):
                scenario = f'm{m}_{correlation}'
                for method in METHODS:
                    subset = data.loc[
                        data['comparison'].eq(comparison)
                        & data['scenario'].eq(scenario)
                        & data['method'].eq(method)
                    ].sort_values('seed')
                    values = subset[metric].to_numpy(dtype=float)
                    assert len(values) == 10
                    position = GROUP_CENTERS[correlation_index] + METHOD_OFFSETS[method]
                    color = METHOD_COLORS[method]

                    ax.boxplot(
                        [values], positions=[position], widths=BOX_WIDTH,
                        patch_artist=True, showfliers=False, manage_ticks=False,
                        boxprops={'facecolor': to_rgba(color, 0.38), 'edgecolor': color, 'linewidth': 0.9},
                        medianprops={'color': '0.12', 'linewidth': 1.0},
                        whiskerprops={'color': color, 'linewidth': 0.8},
                        capprops={'color': color, 'linewidth': 0.8},
                    )

                    rng = np.random.default_rng(
                        stable_jitter_key(metric, comparison, scenario, method)
                    )
                    jitter = rng.uniform(-0.035, 0.035, size=len(values))
                    ax.scatter(
                        position + jitter, values, s=9,
                        color=color, alpha=0.58, linewidths=0, zorder=3,
                    )

            panel_label = chr(ord('a') + panel_index)
            ax.set_title(rf'({panel_label}) $m={m}$', pad=4)
            panel_index += 1
            ax.set_xticks(GROUP_CENTERS, CORRELATION_LABELS)
            ax.set_xlim(-0.48, 2.48)
            ax.set_ylim(0, y_upper)
            ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
            ax.grid(axis='y', alpha=0.20, linewidth=0.5)
            ax.set_axisbelow(True)
            for spine in ax.spines.values():
                spine.set_color('0.35')
                spine.set_linewidth(0.65)

    axes[0, 0].set_ylabel(ylabel)
    axes[1, 0].set_ylabel(ylabel)
    fig.text(0.5, 0.955, ROW_LABELS['complete'], ha='center', va='center', fontsize=10.0)
    fig.text(0.5, 0.515, ROW_LABELS['equal_cardinality'], ha='center', va='center', fontsize=10.0)
    fig.supxlabel('Correlação', y=0.092, fontsize=9.0)

    legend_handles = [
        Patch(
            facecolor=to_rgba(METHOD_COLORS[method], 0.38),
            edgecolor=METHOD_COLORS[method], label=METHOD_LABELS[method],
        )
        for method in METHODS
    ]
    fig.legend(
        handles=legend_handles, loc='lower center',
        bbox_to_anchor=(0.5, 0.018), ncol=4, frameon=False,
    )

    png_path = OUT_DIR / f'{stem}.png'
    pdf_path = OUT_DIR / f'{stem}.pdf'
    fig.savefig(png_path, dpi=300)
    fig.savefig(pdf_path, dpi=300)
    plt.close(fig)
    return png_path, pdf_path

igd_png, igd_pdf = draw_metric('IGD', r'IGD $\downarrow$', 'boxplots_igd_completa_equalizada')
hv_png, hv_pdf = draw_metric('HV', r'HV $\uparrow$', 'boxplots_hv_completa_equalizada')

In [ ]:
metadata = {
    'source': SOURCE_PATH.relative_to(ROOT).as_posix(),
    'methods': METHODS,
    'method_colors': METHOD_COLORS,
    'method_display_labels': METHOD_LABELS,
    'scenarios': [f'm{m}_{correlation}' for m in M_VALUES for correlation in CORRELATIONS],
    'comparisons': COMPARISONS,
    'equalization_method': 'KMeans-medoid',
    'replicates_per_box': 10,
    'individual_points_overlaid': True,
    'publication_width_cm': FIGURE_WIDTH_CM,
}
metadata_path = OUT_DIR / 'boxplots_igd_hv_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')

artifacts = [
    source_export, summary_path,
    igd_png, igd_pdf, hv_png, hv_pdf, metadata_path,
]
for artifact in artifacts:
    assert artifact.exists() and artifact.stat().st_size > 0
print('Arquivos gerados:')
for artifact in artifacts:
    print(' -', artifact.relative_to(ROOT).as_posix())

## Leitura das figuras

As caixas mostram a mediana e o intervalo interquartil, enquanto os bigodes e os dez pontos individuais permitem avaliar dispersão, valores extremos e estabilidade. A separação em duas linhas impede que os resultados das frentes completas sejam confundidos com os resultados obtidos após a equalização de cardinalidade. Assim, a comparação inferior permite verificar diretamente se uma vantagem observada na frente completa depende apenas de um conjunto mais numeroso de soluções.